In [ ]:
library(tidyr)
library(ggplot2)
library(ggh4x)
library(dplyr)
library(RColorBrewer)
library(tidyselect)
library(tidyr)
library(stringr)
library(ggpubr)
library(ggdist)
library(labdsv)
library(vegan)
library(readr)
library(ape)
library(lme4)
library(lsmeans)
library(scales)
library(purrr)
library(broom)
library(Polychrome)
library(cowplot)
library(reshape2)
library(doParallel)
library(Maaslin2)

In [ ]:
metadata <- read.csv("DOME-16S-Nasal-Metadata.csv")
metadata <- metadata %>%
  dplyr::mutate(Sample.ID = stringr::str_remove(Sample.ID, "^19-"))

GENUS <- read.csv("~/projects/Combined-DOME/outputs/DOME-16S-Taxa-Counts-Genus.csv")

asv_counts <- GENUS %>% dplyr::rename(Sample.ID = X)

relative_abundance_genus <- asv_counts %>%
  tibble::column_to_rownames("Sample.ID") %>%
  { 
    rs <- rowSums(.)
    rs[rs == 0] <- 1  # prevent divide by zero
    sweep(., 1, rs, "/")
  } %>%
  as.data.frame() %>%
  tibble::rownames_to_column("Sample.ID")

# Step 3: Apply 0.05% cutoff
relative_abundance_genus <- relative_abundance_genus %>%
  dplyr::mutate(across(-Sample.ID, ~ ifelse(.x <= 0.0005, 0, .x)))

In [ ]:
maaslin2_wide <- relative_abundance_genus %>%
    pivot_longer(-Sample.ID, names_to = "Genus", values_to = "Abundance")

In [ ]:
merged_wide <- maaslin2_wide %>%
  pivot_wider(
    names_from = Genus,
    values_from = Abundance,
    values_fill = 0   # fill missing genus with 0
  )

In [ ]:
# Convert from tibble to dataframe 
abund_matrix <- as.data.frame(merged_wide)        # ensure it’s a data.frame
rownames(abund_matrix) <- abund_matrix$Sample.ID  # assign row names
abund_matrix$Sample.ID <- NULL     

In [ ]:
rownames(metadata) <- metadata$Sample.ID
metadata_matrix <- metadata[rownames(abund_matrix), ]

In [ ]:
# Make sure Cohort is a factor with reference level, same for the other fixed effects
# Order determines which is the reference argument used internally for Maaslin2
metadata_matrix$Year <- factor(metadata_matrix$Year,
                               levels = c("Year 1", "Year 2", "Year 3", "Year 4"))
metadata_matrix$Cohort <- factor(metadata_matrix$Cohort, 
                                 levels = c("Non-Farmer", "Farmer", "Cow"))
metadata_matrix$Season <- factor(metadata_matrix$Season, 
                                 levels = c("Spring", "Summer", "Autumn", "Winter"))
metadata_matrix <- metadata_matrix[match(rownames(abund_matrix), metadata_matrix$Sample.ID), ]

In [ ]:
write.csv(abund_matrix, "MaAsLin2-Abundance-Matrix.csv", row.names = TRUE)

In [ ]:
write.csv(metadata_matrix, "MaAsLin2-Metadata.csv", row.names = TRUE)

In [ ]:
# Cohort
fit_data <- Maaslin2(
  input_data = abund_matrix,        # features x samples
  input_metadata = metadata_matrix,        # metadata with matching rownames
  output = "../Maaslin2/Cohort-Non-Farmer-Reference/",      # output directory
  fixed_effects = c("Cohort"), # adjust for covariates
  random_effects = c("Subject", "Site", "Year", "Season"),
  cores = 8
)

In [ ]:
maaslin2_cohort_results <-read_delim("../Maaslin2/Cohort-Non-Farmer-Reference/significant_results.tsv", 
                         "\t", escape_double = FALSE, trim_ws = TRUE)

# Keep only cohort associations, Cow/Farmer, significant
nasal_ASV_sig <- maaslin2_cohort_results %>%
  filter(metadata == "Cohort") %>%
  filter(value %in% c("Cow", "Farmer")) %>%
  filter(qval < 0.05)

# Keep only features significant in BOTH cohorts
features_both <- nasal_ASV_sig %>%
  group_by(feature) %>%
  filter(n_distinct(value) == 2) %>%
  ungroup() %>%
  pull(feature) %>% unique()

nasal_ASV_both <- nasal_ASV_sig %>%
  filter(feature %in% features_both)

# Select top 50 features by effect size (abs(coef)) or abundance
nasal_ASV_top <- nasal_ASV_both %>%
  group_by(feature) %>%
  summarise(max_coef = max(abs(coef)), .groups = "drop") %>%
  arrange(desc(max_coef)) %>%
  slice_head(n = 75) %>%        # pick top 50 features
  pull(feature)

nasal_ASV_top_cohort <- nasal_ASV_both %>%
  filter(feature %in% nasal_ASV_top)


In [ ]:
write.csv(nasal_ASV_top_cohort, "MaAsLin2-Significant-Hits-For-Cow-And-Farmer.csv", row.names = FALSE)

In [ ]:
palette_colors = c("Cow" = "#990006", "Non-Farmer" = "#B5B5B2", "Farmer" = "#386EC2")

In [ ]:
maaslin2_cohort_plot <- ggplot(nasal_ASV_top_cohort, aes(x = coef, y = reorder(feature, coef), color = value)) +
scale_color_manual(values = palette_colors, name = element_blank(), labels = c("Cow" = "Cows vs Non-Farmers", "Farmer" = "Farmers vs Non-Farmers")) +  # custom text) + 
  geom_pointrange(aes(xmin = coef - stderr, xmax = coef + stderr), shape = 16, size = .4) +
  labs(x = "Coefficient of enrichment", y = "Genus") +
  theme_bw() +
  theme(
    legend.position = c(.98, 0.03),  # inside bottom-right
    legend.justification = c("right", "bottom"), # anchor point
    text = element_text(family = "Helvetica", size = 12),
    legend.margin = margin(2, 10, 5, 8),              # padding inside the box: top, right, bottom, left
    legend.spacing = unit(0.2, "cm"),                # spacing between items
    legend.key.size = unit(0.4, "cm"),               # size of symbols
    legend.text = element_text(size = 8),
    legend.background = element_rect(fill = alpha("white", 0.5), color = "black", linewidth = 0.15), # semi-transparent background
  ) + 
  geom_vline(xintercept = 0, colour = "black", size = 1, linetype = "dotted")

In [ ]:
ggplot2::ggsave(filename = "Maaslin2-Cohort-Non-Farmer-Reference.pdf", maaslin2_cohort_plot, dpi = 320, width = 8, height = 14, units = "in")